# 6. Classification

Predict the 5 NewsGuard levels (logistic regression with balanced classes; ModernBERT with balanced cross-entropy). Same split, same models for both tasks. Linear models: the penalty is chosen with 3 inner folds of the
training outlets. ModernBERT: 5 epochs, lr 3e-5, effective batch 32, last epoch kept. For every test article the
outputs are saved in `predictions/classification/<model>/`, and every fold model in `Modelli/20_FromScratch/`.

In [1]:
TASK = "classification"
MODELS = "TF-IDF,Qwen3-8B,Mistral-24B"     # "ModernBERT" runs on the GPU
N_JOBS = 16

In [2]:
# Parameters
MODELS = "ModernBERT"


In [3]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO

art, outlets = MO.load()

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:
for name in MODELS.split(","):
    for k in range(1, 6):
        if name == "ModernBERT":
            MO.bert_fold(TASK, art, k, "answerdotai/ModernBERT-base", name, LOG)
        else:
            MO.linear_fold(TASK, name, art, k, X=None if name == "TF-IDF" else MO.load_embeddings(name, art),
                           n_jobs=N_JOBS, logfile=LOG)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2026-10-31 00:33:45  classification, ModernBERT, fold 1, epoch 1/5: loss 1.1781 (987 s)


2026-10-31 00:49:53  classification, ModernBERT, fold 1, epoch 2/5: loss 0.6720 (1955 s)


2026-10-31 01:06:16  classification, ModernBERT, fold 1, epoch 3/5: loss 0.5252 (2938 s)


2026-10-31 01:21:58  classification, ModernBERT, fold 1, epoch 4/5: loss 0.4567 (3880 s)


2026-10-31 01:37:11  classification, ModernBERT, fold 1, epoch 5/5: loss 0.4351 (4793 s)


... (25 outputs left out)

2026-10-31 06:13:14  classification, ModernBERT, fold 5, epoch 3/5: loss 0.5319 (2752 s)


2026-10-31 06:28:27  classification, ModernBERT, fold 5, epoch 4/5: loss 0.4614 (3665 s)


2026-10-31 06:43:15  classification, ModernBERT, fold 5, epoch 5/5: loss 0.4382 (4553 s)


2026-10-31 06:44:26  classification, ModernBERT, fold 5: done in 4624 s, GPU peak 10.0 GB


In [5]:
for name in MODELS.split(","):
    folder = C.PREDICTIONS / TASK / name
    if name == "ModernBERT":
        print(pd.concat([pd.read_csv(f).assign(fold=f.stem[-1]) for f in sorted(folder.glob("loss_fold*.csv"))])
              .pivot(index="epoch", columns="fold", values="loss").round(4))
    else:
        p = pd.concat([pd.read_csv(f) for f in sorted(folder.glob("penalty_fold*.csv"))])
        print(name, "chosen penalty per fold:", p[p.chosen].set_index("fold").penalty.to_dict())

fold        1       2       3       4       5
epoch                                        
1      1.1781  1.2356  1.1932  1.1530  1.1839
2      0.6720  0.7237  0.6696  0.6840  0.6882
3      0.5252  0.5530  0.5192  0.5181  0.5319
4      0.4567  0.4709  0.4516  0.4486  0.4614
5      0.4351  0.4403  0.4308  0.4280  0.4382
